# Actor Holdout Feature Stability

Ce notebook reprend le script `actor_holdout_feature_stability.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Verifie la stabilite des mitigations actor-holdout pour le modele live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated-seed actor-held-out feature mitigation stability.
- Commande de reproduction referencee : actor holdout feature stability.
- Artefacts controles : Repeated-seed actor-held-out feature mitigation stability exists. (`runs/exp_037_actor_holdout_feature_stability/metrics/actor_holdout_feature_stability_summary.csv`).
- Run par defaut : `runs/exp_037_actor_holdout_feature_stability`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "actor_holdout_feature_stability.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch

from actor_holdout_feature_ablation import make_actor_split, train_normalize
from actor_holdout_sequence_experiments import selection_score
from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, evaluate_catalogue_model, make_run_dir, set_seed, train_one_model
from sequence_feature_ablation_experiments import build_feature_groups, load_sequence


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics, run_dir):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    rows = []
    group_cols = ["held_actor", "feature_set", "spec", "split"]
    metric_cols = ["average_precision", "roc_auc", "best_hit_rate", "best_false_alarms_per_min", "best_window_precision"]
    for keys, group in h1.groupby(group_cols):
        held_actor, feature_set, spec, split = keys
        row = {
            "held_actor": held_actor,
            "feature_set": feature_set,
            "spec": spec,
            "split": split,
            "n_repeats": int(group["repeat_seed"].nunique()),
            "feature_count": int(group["feature_count"].iloc[0]),
        }
        for col in metric_cols:
            row[f"{col}_mean"] = float(group[col].mean())
            row[f"{col}_std"] = float(group[col].std(ddof=0))
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(run_dir / "metrics" / "actor_holdout_feature_stability_summary.csv", index=False)

    lines = ["# Actor-Held-Out Feature Stability", ""]
    lines.append("Repeated train/val splits for the selected actor-gap mitigation candidates. Held-out actor remains the test domain.")
    lines.append("")
    for held_actor in sorted(summary["held_actor"].unique()):
        lines.append(f"## Held Out: {held_actor}")
        lines.append("")
        test = summary[(summary["held_actor"] == held_actor) & (summary["split"] == "test")].sort_values("average_precision_mean", ascending=False)
        lines.append("| rank | feature set | spec | repeats | AP mean | AP std | ROC AUC | hit | FA/min | precision |")
        lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|---:|")
        for rank, (_, row) in enumerate(test.iterrows(), start=1):
            lines.append(
                f"| {rank} | {row['feature_set']} | {row['spec']} | {int(row['n_repeats'])} | "
                f"{row['average_precision_mean']:.3f} | {row['average_precision_std']:.3f} | "
                f"{row['roc_auc_mean']:.3f} | {row['best_hit_rate_mean']:.3f} | "
                f"{row['best_false_alarms_per_min_mean']:.3f} | {row['best_window_precision_mean']:.3f} |"
            )
        lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- This tests whether the head/torso feature mitigation survives different remaining-actor train/val splits.")
    lines.append("- If the held-out actor still has low AP, the actor robustness limitation remains even if the mitigation improves over all-features.")
    (run_dir / "actor_holdout_feature_stability_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source, X_norm, y, base_meta, feature_columns = load_sequence(args.sequence_run)
    data = np.load(source / "features" / "sequence_dataset.npz")
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw_all = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    videos = pd.read_csv(ROOT / "annotations" / "videos.csv")[["video_id", "actor"]]
    base_meta = base_meta.merge(videos, on="video_id", how="left")
    actors = [actor for actor in sorted(base_meta["actor"].dropna().unique()) if actor]
    all_groups = build_feature_groups(feature_columns)
    groups = {name: all_groups[name] for name in args.groups if name in all_groups}
    spec_map = {
        "tcn_aug_focal": ("tcn_aug_focal", "tcn", True, "focal"),
        "tcn_noaug_bce": ("tcn_noaug_bce", "tcn", False, "bce"),
        "cnn1d_aug_focal": ("cnn1d_aug_focal", "cnn1d", True, "focal"),
    }
    specs = [spec_map[name] for name in args.specs]

    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "source_run": str(source),
            "actors": actors,
            "seeds": args.seeds,
            "feature_groups": {name: [feature_columns[i] for i in idxs] for name, idxs in groups.items()},
            "specs": args.specs,
            "epochs": args.epochs,
            "patience": args.patience,
        },
    )
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    split_rows = []
    for repeat_seed in args.seeds:
        for held_actor in actors:
            split = make_actor_split(base_meta, held_actor, repeat_seed)
            meta = base_meta.copy()
            meta["split"] = meta["video_id"].map(split)
            split_counts = meta.groupby("split")["video_id"].nunique().to_dict()
            split_rows.append({"repeat_seed": repeat_seed, "held_actor": held_actor, **split_counts})
            for group_name, idxs in groups.items():
                X_group_raw = X_raw_all[:, :, idxs].astype(np.float32)
                X_group, split_mean, split_std = train_normalize(X_group_raw, meta)
                np.savez_compressed(
                    run_dir / "features" / f"normalizer_seed{repeat_seed}_holdout_{held_actor}_{group_name}.npz",
                    mean=split_mean,
                    std=split_std,
                )
                for spec_name, kind, augment, loss in specs:
                    set_seed(repeat_seed)
                    model_name = f"seed{repeat_seed}_holdout_{held_actor}_{group_name}_{spec_name}"
                    print(f"training {model_name}")
                    model_args = SimpleNamespace(
                        seed=repeat_seed,
                        batch_size=args.batch_size,
                        lr=args.lr,
                        weight_decay=args.weight_decay,
                        epochs=args.epochs,
                        patience=args.patience,
                        loss=loss,
                        label_smoothing=args.label_smoothing,
                        focal_gamma=args.focal_gamma,
                    )
                    model, history, train_time_s, model_size_bytes = train_one_model(
                        model_name,
                        kind,
                        augment,
                        X_group,
                        y,
                        meta,
                        run_dir,
                        model_args,
                        device,
                    )
                    for row in history:
                        row["repeat_seed"] = repeat_seed
                        row["held_actor"] = held_actor
                        row["feature_set"] = group_name
                        row["feature_count"] = len(idxs)
                        row["spec"] = spec_name
                    all_history.extend(history)
                    rows, _ = evaluate_catalogue_model(
                        model_name,
                        model,
                        X_group,
                        y,
                        meta,
                        run_dir,
                        device,
                        train_time_s,
                        model_size_bytes,
                        args.batch_size,
                        loss,
                    )
                    for row in rows:
                        row["repeat_seed"] = repeat_seed
                        row["held_actor"] = held_actor
                        row["feature_set"] = group_name
                        row["feature_count"] = len(idxs)
                        row["spec"] = spec_name
                        row["selection_score"] = selection_score(row) if row["split"] == "val" and float(row["horizon_s"]) == 1.0 else np.nan
                    all_metrics.extend(rows)
                    pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "actor_holdout_feature_stability_metrics.csv", index=False)
                    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "actor_holdout_feature_stability_training_history.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "actor_holdout_feature_stability_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "actor_holdout_feature_stability_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "actor_holdout_feature_stability_split_counts.csv", index=False)
    summarize(metrics, run_dir)
    append_report(run_dir, "Actor-Held-Out Feature Stability", f"- Summary: `{run_dir / 'actor_holdout_feature_stability_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated-seed actor-held-out feature mitigation stability.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_037_actor_holdout_feature_stability")
    parser.add_argument("--groups", nargs="+", default=["head_torso_geometry_motion", "all_features"])
    parser.add_argument("--specs", nargs="+", default=["tcn_aug_focal", "tcn_noaug_bce", "cnn1d_aug_focal"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[101, 202, 303])
    parser.add_argument("--epochs", type=int, default=12)
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=2e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.08)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_037_actor_holdout_feature_stability_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["actor_holdout_feature_stability.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
